<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w04_baseline_score_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [30]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

Working dir: /content/Applied-Search-Intelligence-System/Applied-Search-Intelligence-System


In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")

In [32]:
df = pd.read_csv(RAW_PATH)
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [33]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [34]:
initial_rows = len(df)
print(f"Initial rows: {initial_rows}")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"After deduplication: {len(df)}")

Initial rows: 30000
After deduplication: 30000


I start from the 30,000-row starter dataset and apply the same population rules used by the starter feature-preparation pipeline: content must have impressions_90d > 0 and content_age_days >= 90. This focuses the analysis on content with observed search demand and enough age to have a meaningful trend signal. I then deduplicate by content_id so that each content item appears once in the analysis population.

In this slice, all 30,000 rows remain after these checks, so the preparation steps do not reduce the population. I keep the checks explicit so the notebook documents exactly which population the baseline is built on.

In [35]:
df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [36]:
# Creating the label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["is_declining_label"].value_counts()

,count
is_declining_label,
1,16262
0,13738


I convert `trend_direction` into a binary `is_declining_label`, where 1 represents a down trend and 0 represents all other trend directions.

In [37]:
base_rate = df["is_declining_label"].mean()
print(f"Base rate (share declining): {base_rate:.3f}")

Base rate (share declining): 0.542


In this analysis population, 54.2% of content items are labeled as declining.

This 54.2% is the base rate. I will use it as the reference point when testing candidate signals: a useful signal should show a meaningfully different decline rate in at least some of its buckets rather than simply reproducing the overall 54.2% rate.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before coding a rule, I check the signals it would lean on. My lane's rule idea is close to the
session's example: **"a page is worth reviewing if it's stale, still gets real traffic, and its
click-through rate is underperforming for its position."** That's three candidate signals:
staleness, volume, and CTR-vs-position, each tied to a real FlyRank flag family (refresh flags,
quick-win, CTR-fix). I check all three below with a bucket table, an `n`, and a one-word verdict,
before deciding how  each one earns a place in the rule.

### 1a. Signal check: staleness (behind the refresh flags)

**Claim:** "Pages that haven't been updated in a long time are more likely to be declining."

**Test:** bucket by `freshness_tier` (derived from `days_since_last_update`), compare
`is_declining_label` mean (decline rate) per bucket against the base rate, with `n` printed.

In [38]:
signal1 = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .sort_index()
)
signal1["decline_rate"] = signal1["decline_rate"].round(3)
print(f"base rate: {base_rate:.3f}\n")
print(signal1)

base rate: 0.542

                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
181+              174         0.471
31-90             175         0.589
91-180           9171         0.611


**Verdict: CONFIRMED (directional, uneven across tiers).**

The two large-`n` buckets tell a clear story: pages updated in the last 30 days sit close to the
base rate (0.511, n=20,480), while pages stuck at 91–180 days since update run meaningfully
higher (0.611, n=9,171), about 10 points above base on a comfortably large sample. That's the
"refresh flag" story confirmed on the buckets that matter.

**The honest negative, and why it doesn't overturn the verdict:** the two edge buckets (31–90
days, n=175; 181+ days, n=174) don't fit a clean "staler = worse" story, 181+ actually sits
*below* base rate (0.471). Both buckets are near the ~50-row sample floor, so I don't trust them
enough to read a reversal into it; more likely this is a survivorship effect (pages nobody has
touched in 6+ months and are *still alive* have often already plateaued rather than being in
active decline) or just noise from a small cell.

**Practical takeaway:** I'll define "stale" as
`days_since_last_update >= 90` (matching the well-populated 91–180 tier), not `>= 180`, using
180 as the threshold would have quietly pulled in the one bucket that actually points the wrong way.

### 1b. Signal check: CTR vs. position (behind the CTR-fix logic)

**Claim:** "Pages getting fewer clicks than typical for their ranking position are more likely to be declining."

**Test:** first check the raw approach, catch a data trap, fix it, then bucket honestly.

In [46]:
# First pass: median CTR per position_tier, all rows.
tier_median_raw = df.groupby("position_tier")["ctr"].median()
print("Raw tier medians (all rows):")
print(tier_median_raw)

Raw tier medians (all rows):
position_tier
deep        0.00
page_1      0.16
page_3_5    0.03
striking    0.11
top_3       0.00
Name: ctr, dtype: float64


 **Trap check:** `avg_position == 0` means "no position data" (per data dictionary), not position zero, but the position_tier bucketing rule (avg_position <= 3) silently sorts every one of those "no data" rows into top_3.

In [49]:
no_pos_in_top3 = ((df["avg_position"] == 0) & (df["position_tier"] == "top_3")).sum()
print(f"No position data' rows hiding inside the top_3 tier: {no_pos_in_top3}")

No position data' rows hiding inside the top_3 tier: 1205


That confirms the trap: **1,205 rows with no real position data are silently sitting inside
`top_3`**, which would corrupt any median or decline-rate read on that tier. I drop rows with
`avg_position == 0` before computing tier medians. I also found that `top_3` and `deep` still
have a **median CTR of 0.00** even after dropping those rows (both tiers are thin — very few
clicks per page — echoing the data dictionary's warning about `top_3`'s tiny volume floor). A
median of 0 makes "below median" mathematically impossible, so I restrict the formal bucket test
to the three tiers with a usable (non-zero) median: `page_1`, `page_3_5`, `striking`.

In [40]:
pos_known = df[df["avg_position"] > 0].copy()
usable_tiers = ["page_1", "page_3_5", "striking"]

tier_median = (
    pos_known[pos_known["position_tier"].isin(usable_tiers)]
    .groupby("position_tier")["ctr"].median()
)
print("Usable-tier medians (no-position rows dropped):")
print(tier_median, "\n")

sub2 = pos_known[pos_known["position_tier"].isin(usable_tiers)].copy()
sub2["tier_median_ctr"] = sub2["position_tier"].map(tier_median)
sub2["ctr_vs_tier"] = np.where(
    sub2["ctr"] < sub2["tier_median_ctr"], "below_tier_median", "at_or_above_tier_median"
)

signal2 = (
    sub2.groupby("ctr_vs_tier")
        .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
)
signal2["decline_rate"] = signal2["decline_rate"].round(3)
print(signal2)
print(f"\nbase rate on this slice: {sub2['is_declining_label'].mean():.3f}")

Usable-tier medians (no-position rows dropped):
position_tier
page_1      0.16
page_3_5    0.03
striking    0.11
Name: ctr, dtype: float64 

                             n  decline_rate
ctr_vs_tier                                 
at_or_above_tier_median  13281         0.564
below_tier_median        13079         0.593

base rate on this slice: 0.578


**Verdict: MIXED.**

Direction is consistent with the claim — pages below their tier's median CTR do decline slightly
more (0.593 vs 0.564, n=13,079 vs 13,281, both far above the sample floor) — but the gap is only
~3 points against a base rate of 0.578. That's a real, large-sample difference, not noise, but it
is a **weak** effect on its own — nowhere near strong enough to gate a rule by itself. This is the
"clearly-explained negative that saves the rule": if I'd gated the rule on CTR-vs-position alone,
I'd have flagged thousands of pages on a ~3-point edge. I'll keep it as a secondary weighting
factor inside the rule (it nudges the score and the reason code), not a primary gate.

### 1c. Signal check: volume (behind quick-win)

**Claim:** "Pages with more current search visibility are more likely to be declining
(there's more to lose)."

**Test:** bucket by `impression_tier`, compare decline rate to base rate, with `n`.

In [41]:
signal3 = (
    df.groupby("impression_tier")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
)
signal3["decline_rate"] = signal3["decline_rate"].round(3)
print(f"base rate: {base_rate:.3f}\n")
print(signal3)

base rate: 0.542

                     n  decline_rate
impression_tier                     
excellent         1078         0.462
good              7205         0.586
low              11248         0.454
moderate         10469         0.615


**Verdict: MIXED / FALSE for the claim as stated.**

There's no clean monotonic story here: `low` (0.454) and `excellent` (0.462) both sit *below*
base rate, while `moderate` (0.615) and `good` (0.586) sit *above* it, all on large, trustworthy
`n`. Volume does not predict decline in a usable direction on its own, this is a clean negative.
What volume *is* good for is something different, a **reliability floor**. The `auditing-signals`
skill's rule is "rates need denominators", a page with 40 impressions can swing from 0% to 100%
CTR on a single click, so any rule that reads CTR needs a volume floor underneath it regardless of
whether volume itself predicts decline. I'll use `impressions_90d >= 500` as a **gate for
measurement trust**, not as a scoring signal.

### My rule, in plain words

A page goes in the refresh-review queue if **(1)** it gets enough real search traffic to trust
its numbers (`impressions_90d >= 500` — the reliability floor from the volume check, not a
decline predictor), **and** **(2)** it has gone at least 90 days without an update (the
staleness threshold that the *large* buckets actually confirmed). Among flagged pages, ones that
*also* show a below-tier-median CTR get a higher score and a different reason code — CTR-vs-
position only earned a supporting role, not a gate, because the signal check showed it's real
but weak.

**Reason codes (one per row):**
- `stale_and_ctr_gap` — visible, stale, and underperforming CTR for its position
- `stale_only` — visible and stale, but CTR is at/above its tier's typical rate (or position
  data isn't usable for the tier-median comparison)
- `not_flagged` — doesn't clear the visibility or staleness gate

**Action label:** `REVIEW_FOR_REFRESH` for any row with a positive score, else `NO_ACTION`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [42]:
# Tier-median CTR, computed only on the three usable tiers (the trap fix from 1b).
# For deep/top_3/no-position rows this is NaN -> ctr_gap is simply False for them,
# never crashes or fabricates a comparison against an unusable median.
usable_tiers = ["page_1", "page_3_5", "striking"]
pos_known_mask = df["avg_position"] > 0
tier_median_map = (
    df.loc[pos_known_mask & df["position_tier"].isin(usable_tiers)]
      .groupby("position_tier")["ctr"].median()
)
df["tier_median_ctr"] = df["position_tier"].map(tier_median_map)

visible = df["impressions_90d"] >= 500          # reliability floor (signal check 1c)
stale = df["days_since_last_update"] >= 90       # confirmed on the large buckets (signal check 1a)
ctr_gap = (
    pos_known_mask
    & df["position_tier"].isin(usable_tiers)
    & (df["ctr"] < df["tier_median_ctr"])
)                                                 # secondary weighting only (signal check 1b)

df["visible"] = visible.astype(int)
df["stale"] = stale.astype(int)
df["ctr_gap"] = ctr_gap.astype(int)

gate = visible & stale
ctr_bonus = np.where(ctr_gap, 0.5, 0.0)          # +50% weight when the secondary signal also fires
df["score"] = np.where(gate, (1 + ctr_bonus) * np.log1p(df["impressions_90d"]), 0.0)

def reason_code(row):
    if row["visible"] == 1 and row["stale"] == 1 and row["ctr_gap"] == 1:
        return "stale_and_ctr_gap"
    if row["visible"] == 1 and row["stale"] == 1:
        return "stale_only"
    return "not_flagged"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = np.where(df["score"] > 0, "REVIEW_FOR_REFRESH", "NO_ACTION")

print(df["reason_code"].value_counts())
print()
print(df["action"].value_counts())

reason_code
not_flagged          23425
stale_only            4556
stale_and_ctr_gap     2019
Name: count, dtype: int64

action
NO_ACTION             23425
REVIEW_FOR_REFRESH     6575
Name: count, dtype: int64


In [43]:
queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

output_columns = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "days_since_last_update", "impressions_90d", "ctr", "avg_position", "position_tier",
    "is_declining_label",  # kept for MY OWN evaluation below -- never used as a rule input
]

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_columns].to_csv(out_path, index=False)
print(f"Wrote {out_path} -- {len(queue)} rows")
queue[output_columns].head(5)

Wrote work/outputs/baseline_action_score.csv -- 30000 rows


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position,position_tier,is_declining_label
0,1,content_5fe46e04994d,client_4e07408562,19.735773,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,517715,0.14,4.2,page_1,1
1,2,content_36ff89c8214e,client_19581e27de,18.892594,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,295097,0.05,7.3,page_1,0
2,3,content_c8e9d6ab9013,client_19581e27de,18.372829,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,208678,0.00,9.7,page_1,1
3,4,content_a7427266c305,client_19581e27de,18.317426,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,201111,0.11,5.7,page_1,0
4,5,content_91652435f57a,client_19581e27de,17.970554,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,159590,0.06,7.8,page_1,0


**Quick honest check against the label** (evaluation only — the label never touches the
score above). Precision@K on the same slice, next to the base rate a random ranking would give:

In [44]:
def precision_at_k(ranked_df, k):
    return ranked_df.head(k)["is_declining_label"].mean()

for k in [20, 50, 100]:
    print(f"precision@{k}: {precision_at_k(queue, k):.3f}   (base rate: {base_rate:.3f})")

from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(df), 1)), df["is_declining_label"])
print(f"\ndummy (majority-class) accuracy floor: {dummy.score(np.zeros((len(df),1)), df['is_declining_label']):.3f}")

precision@20: 0.650   (base rate: 0.542)
precision@50: 0.520   (base rate: 0.542)
precision@100: 0.530   (base rate: 0.542)

dummy (majority-class) accuracy floor: 0.542


precision@20 (0.65) clears the base rate by a real margin; precision@50 and precision@100
basically flatten back out to base rate. That's an honest, useful finding for week 5: **the rule
is only sharp at the very top of the queue** — exactly where the biggest, stalest, CTR-
underperforming pages sit. Past the top ~20–30 it's barely better than random. This is the number
next week's model has to beat, and *where* it needs to beat it (mid-queue, not just top-20).

## 3. Top-20 review

*For each of your top 20: action, reason code, confidence note, and what would make it wrong.*

Reading the raw top 20 by rank first, then the honest read below it.

In [45]:
top20 = queue[output_columns].head(20)
top20

,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position,position_tier,is_declining_label
0,1,content_5fe46e04994d,client_4e07408562,19.735773,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,517715,0.14,4.2,page_1,1
1,2,content_36ff89c8214e,client_19581e27de,18.892594,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,295097,0.05,7.3,page_1,0
2,3,content_c8e9d6ab9013,client_19581e27de,18.372829,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,208678,0.00,9.7,page_1,1
3,4,content_a7427266c305,client_19581e27de,18.317426,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,201111,0.11,5.7,page_1,0
4,5,content_91652435f57a,client_19581e27de,17.970554,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,159590,0.06,7.8,page_1,0
5,6,content_f42eb861c6dd,client_19581e27de,17.902065,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,152467,0.13,6.5,page_1,1
6,7,content_11fcfd65d94c,client_19581e27de,17.868398,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,149083,0.15,6.2,page_1,1
7,8,content_97a86caf3a3d,client_19581e27de,17.854113,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,147670,0.07,6.4,page_1,1
8,9,content_8b36799b7e44,client_6208ef0f77,17.789033,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,141400,0.02,32.0,page_3_5,1
9,10,content_c1fe78bc4e37,client_19581e27de,17.709019,stale_and_ctr_gap,REVIEW_FOR_REFRESH,104,134055,0.03,7.5,page_1,1


**Row-by-row read (rank 1–20):**

All 20 top rows share `reason_code = stale_and_ctr_gap`, `action = REVIEW_FOR_REFRESH`, and
(strikingly) `days_since_last_update = 104` for every single one — a sign these pages were
likely refreshed in the same batch/migration, not that 104 is some magic threshold. Ranks 1–8
and 14–20 sit in `page_1` (positions ~4–9), ranks 9–13 sit in `page_3_5` (positions ~24–47,
still comfortably in the striking-to-page-3 range) — every row is a genuinely large, visible
page (95K–518K impressions/90d), so the visibility gate is doing its job.

- **Why each is here:** all 20 cleared the visibility floor (≥500 impressions — these are all
  orders of magnitude above that), cleared the 90-day staleness gate at exactly 104 days since
  update, and additionally scored below their position tier's median CTR — the combination that
  earns the `stale_and_ctr_gap` reason code and the top slots (the CTR bonus plus log(impressions)
  push these to rank 1).
- **What would make each wrong:** for rank 1 (content_5fe46e04994d, ctr=0.14 vs tier median 0.16,
  page_1) — a near-median CTR this close could just be normal week-to-week noise on a busy page,
  not a real underperformance; if it's a highly seasonal page catching a normal seasonal dip,
  "stale" and "underperforming" are both the wrong read. For ranks 3, 9–13 (ctr near 0.00–0.03,
  page_3_5 positions in the 20s–40s) — a near-zero CTR at position 24–47 may simply be typical
  for that position (CTR naturally craters past page 1), not a fixable content problem; the
  tier-median comparison partially controls for this, but a single global median per tier still
  blends a wide range of true positions.
- **General pattern across all 20:** because score = `(1 + ctr_bonus) * log1p(impressions)`, huge
  page_1 pages with borderline CTR gaps dominate the top purely by scale — the actual size of the
  CTR shortfall matters less than I'd like. That's a real weakness I flag properly in section 4.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks in the top 20, and why:**

1. **Rank 1 (content_5fe46e04994d)** — CTR is 0.14 against a tier median of 0.16: an 0.02-point
   gap. That is inside likely day-to-day noise for a page pulling 500K+ impressions/90d, not a
   meaningful underperformance signal. My rule treats "below median" as binary, so a 0.02-point
   miss scores identically to a 0.13-point miss (rank 3, ctr=0.00 vs median 0.16). **What would
   make it wrong:** if this page's CTR has been stable at ~0.14 for months (i.e. this is just its
   normal rate, not a decline), the `ctr_gap` reason code is misleading — it should really just be
   `stale_only`.
2. **Ranks 9, 11–13 (page_3_5 tier, positions 24–47)** — the `page_3_5` tier spans a huge range of
   true positions (page 3 through page 5), and CTR falls off steeply and non-linearly across that
   range. Comparing a position-47 page's CTR to the *same* tier median as a position-24 page
   understates how "expected" its low CTR actually is. **What would make it wrong:** if
   position 47 pages typically get near-zero CTR anyway, flagging this page for a "CTR fix" sends
   an editor chasing a problem that isn't fixable by content changes.
3. **The `days_since_last_update == 104` clustering across all 20 rows** — this isn't a single
   bad pick, but a structural weak spot: my rule can't currently distinguish "104 days, just over
   the line" from "800 days, badly neglected" once both clear the same 90-day gate and the same
   500-impression floor — score is driven almost entirely by `log1p(impressions)` at that point.
   **What would make the whole rule wrong:** if this repo's `days_since_last_update` values are
   themselves an artifact of a bulk migration/export rather than real edit history, the staleness
   gate is measuring the export, not staleness.

**No-leakage check:**
- Rule inputs used: `impressions_90d`, `days_since_last_update`, `ctr`, `avg_position`,
  `position_tier`. None of these are derived from `trend_direction` or `trend_pct`.
- `trend_direction` / `trend_pct` / `is_declining_label` are **not** read anywhere inside the
  `score`, `reason_code`, or `action` logic above — `is_declining_label` only appears in the
  evaluation cell (precision@K), after the queue is already ranked.
- No 30-day trend windows (`impressions_last_30d`, `impressions_prev_30d`) are used as rule
  inputs — those are exactly the columns that build the label, so they're excluded on purpose.
- All features are current-state snapshots (trailing-90-day totals or point-in-time tiers), not
  future-looking windows.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.